<a href="https://colab.research.google.com/github/IT21280238/Emotion-Recognition-Intensity-Prediction-/blob/Arousal-Valance-Prediction/Affect_Reg_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
df_4= pd.read_csv("/content/expression_4.csv")

In [ ]:
df_4.info()

In [ ]:
df_4.head()

In [ ]:
df_4_val= pd.read_csv("/content/expression_val_4.csv")

In [ ]:
df_4_val.head()

In [ ]:
# Drop unnecessary columns
df_4 = df_4.drop(columns=["Expression", "Index"])

# Save the cleaned dataset
df_4.to_csv("df_4.csv", index=False)

# Display the first few rows of the cleaned dataset
print(df_4.head())


In [ ]:
# Drop unnecessary columns
df_4_val = df_4_val.drop(columns=["Expression", "Index"])

# Save the cleaned dataset
df_4_val.to_csv("df_4_val.csv", index=False)

# Display the first few rows of the cleaned dataset
print(df_4_val.head())


In [ ]:
from matplotlib import pyplot as plt
df_4['Arousal'].plot(kind='hist', bins=20, title='Arousal')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
df_4['Valence'].plot(kind='hist', bins=20, title='Valence')
plt.gca().spines[['top', 'right',]].set_visible(False)

**Regression**

In [ ]:
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input

def define_mtl_va_model(input_shape=(224, 224, 3)):
    inputs = Input(shape=input_shape)

    # Shared Feature Extraction
    x = Conv2D(64, (7,7), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)

    x = Conv2D(128, (3,3), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)

    x = Conv2D(256, (1,1), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)

    x = Conv2D(256, (3,3), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), activation='relu', kernel_initializer='he_uniform',
               padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.4)(x)

    x = Flatten()(x)

    # Task-Specific Layers
    valence_branch = Dense(256, activation='relu', kernel_initializer='he_uniform',
                           kernel_regularizer=regularizers.l2(0.001))(x)
    valence_branch = BatchNormalization()(valence_branch)
    valence_branch = Dropout(0.5)(valence_branch)
    valence_output = Dense(1, activation='linear', name='valence')(valence_branch)

    arousal_branch = Dense(256, activation='relu', kernel_initializer='he_uniform',
                           kernel_regularizer=regularizers.l2(0.001))(x)
    arousal_branch = BatchNormalization()(arousal_branch)
    arousal_branch = Dropout(0.5)(arousal_branch)
    arousal_output = Dense(1, activation='linear', name='arousal')(arousal_branch)

    # Define Model with Two Outputs
    model = Model(inputs=inputs, outputs=[valence_output, arousal_output])

    return model


In [ ]:
model = define_mtl_va_model(input_shape=(224, 224, 3))

In [ ]:
import cv2
import numpy as np
import os

# Convert JPG images to NPY format
for img_path in df_0_val["Image"]:
    img = cv2.imread(img_path)  # Read image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    img = cv2.resize(img, (224, 224))  # Resize
    img = img.astype(np.float32) / 255.0  # Normalize

    npy_path = img_path.replace(".jpg", ".npy")  # Save as .npy
    np.save(npy_path, img)

    # Update dataset path to the new .npy file
    df_4_val["Image"] = df_4_val["Image"].replace(img_path, npy_path)

# Save the updated CSV file
df_4_val.to_csv("df_4_val_updated.csv", index=False)


In [ ]:
df_0_val.head()

In [ ]:
def load_and_preprocess(img_path, valence, arousal):
    def _load_npy(path):
        path = path.numpy().decode("utf-8")  # Convert Tensor to string
        img = np.load(path)  # Load .npy image
        img = cv2.resize(img, (224, 224))  # Resize

        # Ensure image has 3 channels
        if len(img.shape) == 2:  # Grayscale image
            img = np.stack([img] * 3, axis=-1)  # Convert to 3-channel format
        elif img.shape[-1] != 3:  # Incorrect channel count
            img = img[:, :, :3]  # Keep only first 3 channels if needed

        img = img / 255.0  # Normalize
        return img.astype(np.float32)

    img = tf.py_function(func=_load_npy, inp=[img_path], Tout=tf.float32)

    # 🚀 Explicitly set shape to avoid TensorFlow shape errors
    img.set_shape((224, 224, 3))

    return img, {"valence": tf.reshape(valence, [1]), "arousal": tf.reshape(arousal, [1])}


In [ ]:
# Convert DataFrame columns to a list
image_paths = df_4["Image"].tolist()
valence_values = df_4["Valence"].tolist()
arousal_values = df_4["Arousal"].tolist()

In [ ]:
# Create Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((image_paths, valence_values, arousal_values))
train_dataset = train_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(8).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Extract validation image paths and labels
val_image_paths = df_4_val["Image"].values  # Paths to validation .npy files
val_valence_labels = df_4_val["Valence"].values.astype(np.float32)
val_arousal_labels = df_4_val["Arousal"].values.astype(np.float32)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((val_image_paths, val_valence_labels, val_arousal_labels))
val_dataset = val_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(8).prefetch(tf.data.AUTOTUNE)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss={'valence': 'mse', 'arousal': 'mse'},
              metrics={'valence': 'mae', 'arousal': 'mae'})

In [ ]:
history = model.fit(train_dataset, validation_data=val_dataset, epochs=10)